# Volatility Research — Hypothesis Tests

Five high-value volatility hypotheses, each a thin wrapper around a trusted
library (`scipy` / `statsmodels`) in `src/hypotheses.py`. Every test returns a
uniform result dict and degrades gracefully when data is insufficient.

| # | Hypothesis | Library test |
|---|-----------|--------------|
| 1 | Leverage effect | Mann-Whitney U (one-sided) |
| 2 | Cross-sector contagion | Granger causality (statsmodels) |
| 3 | Variance risk premium | Mann-Whitney U (one-sided) |
| 4 | Earnings-week vol premium | Permutation test |
| 5 | Regime-dependent leverage | scipy bootstrap CI + permutation |


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys, os
sys.path.insert(0, os.path.abspath('.'))

import pandas as pd
import numpy as np

from src.data_loader import load_stock_data
from src.data_helpers import load_vix, load_earnings_dates
from src import hypotheses as H

START   = '2015-01-01'
END     = '2024-12-31'
PRIMARY = 'MU'

# Tickers spanning three DEFAULT_SECTORS sectors for the contagion test.
TICKERS = ['MU', 'NVDA', 'JPM', 'XOM']

print('Loading data...')
# Each frame already carries log_return and realized_vol_21d.
df_dict = {t: load_stock_data(t, START, END) for t in TICKERS}

# Primary single-ticker frame, augmented with a VIX level for the VRP test.
df = df_dict[PRIMARY].copy()
df['vix_level'] = load_vix(START, END).reindex(df.index, method='ffill')

earnings = load_earnings_dates(PRIMARY)
print('Setup complete:', {t: len(v) for t, v in df_dict.items()})


## H1 — Leverage effect

Negative-return days are followed by higher realized vol than positive-return
days (Black 1976). One-sided Mann-Whitney U with a rank-biserial effect size.


In [ ]:
r1 = H.test_leverage_effect(df)
print(r1['conclusion'])


## H2 — Cross-sector contagion

Directed Granger-causality matrix over sector-average vol series, ranking
sectors by net contagion (significant outgoing − incoming links).


In [ ]:
r2 = H.test_cross_sector_contagion(df_dict)
print(r2['conclusion'])


## H3 — Variance risk premium

A negative variance risk premium (RV > VIX) predicts higher forward vol.
Forward vol is split by VRP sign and compared with a one-sided Mann-Whitney U.


In [ ]:
r3 = H.test_variance_risk_premium(df, ticker=PRIMARY, horizon=10)
print(r3['conclusion'])


## H4 — Earnings-week vol premium

Realized vol is higher in the ±2 days around earnings than on other days.
Permutation test on the difference in means (earnings windows are rare events).


In [ ]:
r4 = H.test_earnings_vol_premium(df, earnings)
print(r4['conclusion'])


## H5 — Regime-dependent leverage

Does the leverage effect amplify in stressed regimes (Campbell & Hentschel
1992)? Per-regime neg/pos forward-vol ratio, a scipy bootstrap CI on the
Extreme-regime ratio, and a permutation test for Extreme ratio > Low ratio.


In [ ]:
r5 = H.test_regime_dependent_leverage(df, ticker=PRIMARY)
print(r5['conclusion'])


## Summary

Collect the five results into one table via `H.compile_summary` and persist a
markdown version to `results_summary.md`.


In [ ]:
results = {
    'leverage_effect': r1,
    'cross_sector_contagion': r2,
    'variance_risk_premium': r3,
    'earnings_vol_premium': r4,
    'regime_dependent_leverage': r5,
}
summary = H.compile_summary(results)
display(summary)

lines = ['# Hypothesis Test Results\n',
         f'Primary ticker: {PRIMARY} | Period: {START} to {END}\n\n',
         '| Hypothesis | Title | p-value | Effect | Significant? |\n',
         '|---|---|---|---|---|\n']
for hyp, row in summary.iterrows():
    lines.append(f"| {hyp} | {row['title']} | {row['p_value']} | "
                 f"{row['effect']} | {row['significant']} |\n")
with open('results_summary.md', 'w', encoding='utf-8') as f:
    f.writelines(lines)
print(''.join(lines))
